# Mai Chi: clone and run on Colab

**Teacher:** pinned ZeroTTS, fixed `maichi` voice. **Student:** compact VITS with an optional F0/voicing predictor and conditioning, 24 kHz waveform output. This is **sequence-level distillation from teacher audio**, inspired by the small single-voice objective of Teacher in the Loop; it does **not** reproduce that paper's Kokoro feature/head/excitation losses or student architecture.

The notebook downloads models and benchmark data, selects text without downloading ZeroSpeech audio, synthesizes a Mai Chi corpus, extracts F0, trains a student, resumes checkpoints, exports inference weights and evaluates teacher versus student. This compact notebook clones `Huyvux12/Ancaptritue` and executes its readable Python source. It records the exact source commit and pipeline checksum. The complete-source alternative is `MaiChi_Standalone_Colab.ipynb`.

**Validation performed before delivery:** real ZeroTTS CPU synthesis on three sentences; waveform checks; real-WAV feature extraction; three optimization steps including GAN, pitch and KL losses; checkpoint resume; deployment reload and inference. These checks do not establish student quality or GPU compatibility. The full GPU run and 137-item quality benchmark have not been completed.

Run the pilot first. Then set `MODE = "full"`, rerun configuration and subsequent cells, and keep the same full run directory when resuming. A pilot model is not expected to be publishable.

## 1. Runtime and experiment configuration

Select a GPU runtime. If you use an external RTX PRO 6000 96 GB through Colab, the GPU must actually be visible to the kernel. The notebook checks this instead of assuming the GPU model. PyTorch must support its CUDA architecture; do not install an old CUDA wheel on a Blackwell GPU.

`pilot`: 512 sentences / 2,000 steps, pipeline trial. `full`: 20,000 sentences / 100,000 steps per student. These are starting budgets, not convergence guarantees. Default batch size is 32; increase to 64 only after inspecting peak memory. A large VRAM budget does not remove the sequential teacher-generation bottleneck.

Keep at least **60 GB free** for a full run with cached features/checkpoints. The actual requirement varies with duration and compression. Drive must have its own free quota. For a remote/local Colab runtime, set `USE_DRIVE=False` and use a persistent `PERSISTENT_BASE`.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json

MODE = "pilot"                 # "pilot" or "full"
USE_DRIVE = True                # False for an external/local Jupyter runtime
PERSISTENT_BASE = "/content/maichi_research_v01"  # Set a persistent disk path when USE_DRIVE=False
TEACHER_DEVICE = "cuda"         # Explicitly change to "cpu" if ORT CUDA cannot run
TRAIN_DEVICE = "cuda"
CPU_THREADS = 4
BATCH_SIZE = 32
FEATURE_WORKERS = 4
RUN_ASR_AUDIT = True
RUN_ABLATION = False             # Also train the same-backbone model without pitch
RUN_MEDIUM = False               # Also train ~18M-deployment-parameter student
RUN_BENCHMARK = MODE == "full"  # Full 137-item evaluation after training
RUN_UTMOS = False                # Optional dependency; skipped values remain missing

assert MODE in {"pilot", "full"}
TEXT_COUNT = 512 if MODE == "pilot" else 20000
TRAIN_STEPS = 2000 if MODE == "pilot" else 100000
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSISTENT_BASE = "/content/drive/MyDrive/maichi_research_v01"
ROOT = Path(PERSISTENT_BASE) / MODE
ROOT.mkdir(parents=True, exist_ok=True)
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
VITS_DIR = WORK / "maichi_vits_pinned"
os.chdir(WORK)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print({"root": str(ROOT), "mode": MODE, "sentences": TEXT_COUNT, "steps": TRAIN_STEPS})
print("Free local GiB:", round(shutil.disk_usage(WORK).free / 2**30, 1))


## Clone the project

`REPO_REF` can be `main`, a branch, tag or commit SHA. The exact checked-out SHA and pipeline checksum are recorded under `ROOT`. Rerunning this cell updates a clean checkout to the requested ref; it refuses to overwrite local edits. To reproduce an earlier run, set `REPO_REF` to the SHA in `source_revision.json`.

This is a public research repository; cloning needs no GitHub token. Models, generated data and checkpoints stay outside the checkout.

In [ ]:
REPO_URL = "https://github.com/Huyvux12/Ancaptritue.git"
REPO_REF = "main"  # Set a commit SHA for an exact reproduction.
REPO_DIR = WORK / "Ancaptritue"

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    assert remote.removesuffix(".git") == REPO_URL.removesuffix(".git"), "Checkout belongs to a different repository."
    dirty = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
    assert not dirty.strip(), "Checkout has local edits. Preserve them before updating, or choose another REPO_DIR."
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
REPO_SHA = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
import hashlib
PIPELINE_SHA = hashlib.sha256((REPO_DIR / "maichi_pipeline.py").read_bytes()).hexdigest()
revision_path = ROOT / "source_revision.json"
if revision_path.exists():
    previous = json.loads(revision_path.read_text())
    assert previous["pipeline_sha256"] == PIPELINE_SHA, "Pipeline changed since this run. Use the recorded REPO_REF or a new run directory."
revision_path.write_text(json.dumps({"repository": REPO_URL, "requested_ref": REPO_REF,
                                    "commit": REPO_SHA, "pipeline_sha256": PIPELINE_SHA}, indent=2))
print("Source commit:", REPO_SHA)


## 2. Install compatible dependencies

The existing CUDA PyTorch installation is retained. `zerotts` is installed without dependencies so it does not add the CPU ONNX package beside the GPU package. Python dependencies are pinned; the notebook also saves a complete package freeze and actual GPU identity. Installation must run before importing NumPy, ONNX Runtime or the pipeline. If a package was imported earlier in this session, restart the runtime once and rerun from the top.

The `setuptools` pin retains `pkg_resources` required by WORLD. The transcript reader uses revision-checked JSON pages to avoid Parquet streaming finalization issues. WORLD is built against the active NumPy.

In [ ]:
def pip_install(*packages, extra=None):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(packages)
    if extra:
        cmd += list(extra)
    subprocess.run(cmd, check=True)

pip_install("setuptools==80.9.0", "wheel", "cython==3.0.12", "numpy==2.2.6")
pip_install(extra=["-r", str(REPO_DIR / "requirements-colab.txt")])
os.environ["CC"], os.environ["CXX"] = "gcc", "g++"
pip_install("pyworld==0.3.5", extra=["--no-build-isolation"])
pip_install("zerotts==0.1.5", extra=["--no-deps"])
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu"], check=True)
if TEACHER_DEVICE == "cuda":
    pip_install("onnxruntime-gpu>=1.23.2,<2")
else:
    pip_install("onnxruntime==1.23.2")


In [ ]:
import torch, numpy as np, onnxruntime as ort
assert torch.cuda.is_available(), "No GPU visible to this kernel. Connect the intended GPU runtime first."
# Real CUDA operations catch missing kernel/architecture support early.
x = torch.randn(512, 512, device="cuda")
_ = x @ x
_ = torch.nn.Conv1d(16, 16, 3).cuda()(torch.randn(2, 16, 128, device="cuda"))
torch.cuda.synchronize()
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Torch/CUDA:", torch.__version__, torch.version.cuda)
print("ONNX providers:", ort.get_available_providers())
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()  # Reuse compatible CUDA/cuDNN libraries loaded by PyTorch.
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
(ROOT / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


## 3. Load the cloned pipeline

The source is `maichi_pipeline.py` in the checkout. VITS is fetched at its pinned commit. Student deployment sizes are measured later and include waveform generation. The complete-source notebook remains available in the repository if you prefer an embedded version.

In [ ]:
sys.path.insert(0, str(REPO_DIR))
sys.modules.pop("maichi_pipeline", None)
import maichi_pipeline as mp
assert Path(mp.__file__).resolve() == (REPO_DIR / "maichi_pipeline.py").resolve()
mp.bootstrap(VITS_DIR)
shutil.copy2(REPO_DIR / "maichi_pipeline.py", ROOT / "maichi_pipeline.py")
shutil.copy2(REPO_DIR / "LICENSE_VITS.txt", ROOT / "LICENSE_VITS.txt")

def run(stage, **kwargs):
    cmd = [sys.executable, str(REPO_DIR / "maichi_pipeline.py"), stage, "--root", str(ROOT), "--vits-dir", str(VITS_DIR)]
    for key, value in kwargs.items():
        flag = "--" + key.replace("_", "-")
        if isinstance(value, bool):
            if value:
                cmd.append(flag)
        elif value is not None:
            cmd += [flag, str(value)]
    print("Running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


## 4. Download ZeroTTS and verify Mai Chi

This stage must succeed before corpus generation. It downloads roughly 0.9 GB of pinned inference assets, loads the `maichi` preset, synthesizes three sentences and checks duration, finite values, RMS and clipping. Listen to the audio too: waveform checks alone cannot verify Vietnamese pronunciation or voice quality.

If ONNX CUDA fails, change `TEACHER_DEVICE` to `cpu`, rerun configuration and continue. Training still uses CUDA. The pipeline refuses a missing CUDA provider instead of silently claiming GPU acceleration; some individual ONNX graph nodes may still execute on CPU even when the provider is registered.

In [ ]:
run("smoke", device=TEACHER_DEVICE, threads=CPU_THREADS)
from IPython.display import Audio, display
for row in json.loads((ROOT / "smoke.json").read_text())["samples"]:
    print(row["text"], "RTF:", round(row["rtf"], 3))
    display(Audio(filename=row["wav"]))


## 5. Reserve the entire ZeroBench test set, then collect training text

The pinned benchmark contains 137 items. It is reserved **before** selecting the corpus. Training text is deduplicated, screened against benchmark targets/reference transcripts and split deterministically by normalized text hash (90/5/5). This does not prove absence of paraphrases or shared source documents; that limitation is recorded.

ZeroSpeech text is sampled from Hugging Face viewer JSON pages. The cached asset revision in every page is checked against the pinned dataset revision before any transcript is accepted. Audio URLs are discarded without fetching them; all accepted text is regenerated using Mai Chi. Vietnamese and mixed Vietnamese/English examples are retained. No 1.7 TB dataset download is requested.

ZeroSpeech and ZeroBench are CC-BY-NC-4.0. This notebook is a research workflow; MIT model code does not remove data/source/voice restrictions.

In [ ]:
run("benchmark")
run("texts", count=TEXT_COUNT)
print(json.loads((ROOT / "data_provenance.json").read_text()))


## 6. Generate the Mai Chi corpus (resumable)

Accepted examples are saved immediately as 24 kHz PCM WAV plus provenance sidecars. A rerun skips matching completed examples. Failed duration, silence, repetition-rate or clipping checks are logged. This filter is not an ASR quality guarantee.

The generator currently uses ZeroTTS's public single-request inference loop. GPU generation speed must be measured on your hardware; do not extrapolate throughput from 96 GB VRAM. Feature extraction below uses CPU workers.

In [ ]:
run("generate", device=TEACHER_DEVICE, threads=CPU_THREADS)
print(json.loads((ROOT / "corpus_summary.json").read_text()))


## 7. Corpus audit and acoustic features

The optional small PhoWhisper audit is separate from the official benchmark: it retains Vietnamese diacritics, records both transcripts and flags potential defects for listening. Its WER is not a leaderboard score and is not used to change test items.

WORLD extracts F0 and voiced/unvoiced labels from teacher audio. Within-token position features let the predictor represent rising and dipping pitch contours. The student learns F0 and voicing together with duration, KL, spectral reconstruction, feature matching and adversarial losses. Pitch conditioning transitions from observed to predicted values over 20,000 steps. Removing the pitch module is the controlled ablation. F0 supervision is not itself a complete lexical-tone recognizer.

In [ ]:
if RUN_ASR_AUDIT:
    run("audit", count=32 if MODE == "full" else 8, device="cuda")
    import pandas as pd
    display(pd.read_json(ROOT / "corpus_asr_audit.jsonl", lines=True).sort_values("wer", ascending=False).head(8))
run("features", workers=FEATURE_WORKERS)
print(json.loads((ROOT / "vocab_audit.json").read_text()))


In [ ]:
vocab = json.loads((ROOT / "vocab.json").read_text())
for size in ["small", "medium"]:
    model = mp.make_student(VITS_DIR, len(vocab), size=size, pitch=True)
    print(size, mp.parameter_counts(model))
    del model


## 8. Train the student and optional ablations

Training uses the available CUDA GPU, BF16 where supported, atomic checkpoints and the same seeds/data for the pitch ablation. `best.pt` is selected by a validation reconstruction loss; this is not an end-to-end speech-quality metric. `last.pt` retains generator, discriminator, optimizers, scaler, RNG state and step. Rerun with the same name/config to resume; the data iterator restarts, so resume is not bit-exact batch replay. Avoid simultaneously running two copies against one directory.

A three-step CPU smoke check is included in the delivery evidence. It verifies plumbing, not convergence. The default pilot is also a plumbing experiment; evaluate its actual audio without expecting natural speech. A full run may need more data or steps than the starting budget.

In [ ]:
EXPERIMENTS = [("small_pitch", "small", True)]
if RUN_ABLATION:
    EXPERIMENTS.append(("small_no_pitch", "small", False))
if RUN_MEDIUM:
    EXPERIMENTS.append(("medium_pitch", "medium", True))
for name, size, with_pitch in EXPERIMENTS:
    run("train", name=name, size=size, no_pitch=not with_pitch, steps=TRAIN_STEPS,
        batch_size=BATCH_SIZE, workers=FEATURE_WORKERS, device=TRAIN_DEVICE, precision=PRECISION,
        save_every=500 if MODE == "pilot" else 1000,
        validate_every=500 if MODE == "pilot" else 1000,
        allow_small=MODE == "pilot")


## 9. Inspect learning curves and free-running audio

Only text enters free-running inference; no teacher duration/F0/audio is passed at inference. Each deployment file includes the text model, duration model, flow, pitch modules and waveform generator. It excludes the posterior encoder and GAN discriminator. No ONNX/INT8 quality claim is made in this version.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
for name, _, _ in EXPERIMENTS:
    history = pd.read_json(ROOT / "runs" / name / "train.jsonl", lines=True)
    history.plot(x="step", y=["mel", "kl", "duration", "pitch"], subplots=True, figsize=(10, 8), title=name)
    plt.tight_layout(); plt.show()
    generator, vocabulary = mp.load_deployment(ROOT / "runs" / name / "student_deploy.pt", VITS_DIR, TRAIN_DEVICE)
    text = "Xin chào, hôm nay chúng ta cùng tìm hiểu trí tuệ nhân tạo."
    ids = mp.encode(text, vocabulary)[None].to(TRAIN_DEVICE)
    wav = generator.speak(ids, torch.tensor([ids.shape[1]], device=TRAIN_DEVICE))[0, 0].float().cpu().numpy()
    display(Audio(wav, rate=mp.SR))
    del generator
    torch.cuda.empty_cache()


## 10. Generate all 137 teacher/student benchmark WAVs

This is an **adapted fixed-Mai-Chi protocol**, not the original zero-shot voice-cloning benchmark. `cross_lingual` still labels the source subset, but no foreign speaker is cloned. Every model receives raw orthography. The original reference-speaker similarity is invalid for this setup and must not be reported as Mai Chi similarity.

Keep speed measurements on the same device/thread configuration. For a CPU speed comparison, rerun synthesis for both systems with `BENCH_DEVICE="cpu"`; the WAVs and timing files are overwritten for that run, so preserve earlier timing files separately if needed. Native synthesis rates differ (teacher 48 kHz, student 24 kHz); report that explicitly. Scorers resample audio consistently for ASR/verifier inputs.

In [ ]:
BENCH_DEVICE = "cuda"  # "cpu" gives a same-CPU comparison; teacher must have working ORT CUDA for "cuda"
if RUN_BENCHMARK:
    run("synthesize", model="teacher", device=BENCH_DEVICE, threads=CPU_THREADS)
    for name, _, _ in EXPERIMENTS:
        run("synthesize", name=name, model="student", device=BENCH_DEVICE, threads=CPU_THREADS)
else:
    print("Benchmark skipped for pilot. Set RUN_BENCHMARK=True when ready.")


## 11. Official ASR, naturalness and silence scoring; adapted speaker similarity

The official scorer uses both `openai/whisper-large-v3` and `vinai/PhoWhisper-large`, expanded references and their minimum WER. Per-sample transcripts and matching references are retained. This is a heavier download than the TTS model itself. Optional UTMOSv2 is disabled by default; skipped metrics remain missing, never zero.

The raw official report also computes similarity against the original diverse reference speakers. **Ignore that field here.** A separate same-text teacher-Mai-Chi versus student WavLM cosine is computed and clearly labeled. It cannot be compared directly to the published zero-shot speaker-similarity score. UTMOS is an automatic estimate; human listening remains necessary.

In [ ]:
if RUN_UTMOS and RUN_BENCHMARK:
    # Optional upstream package may constrain Torch/timm. Inspect pip's plan if your runtime has conflicts.
    pip_install("git+https://github.com/sarulab-speech/UTMOSv2.git")
    (ROOT / "pip_freeze_with_utmos.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
if RUN_BENCHMARK:
    for name in ["teacher_maichi"] + [x[0] for x in EXPERIMENTS]:
        run("score", name=name, device="cuda", with_utmos=RUN_UTMOS)
    for name, _, _ in EXPERIMENTS:
        run("similarity", name=name, device="cuda")
    run("report")
    display(pd.read_csv(ROOT / "comparison.csv"))
    print("Per-sample official quality reports:", ROOT / "scores")


## 12. Listening-test worksheet and result locations

The worksheet below creates randomized A/B pairs from 30 unique benchmark texts. The key is separate from the rating sheet. Ask listeners to judge intelligibility, tone mistakes, naturalness and Mai Chi similarity. Do not infer human MOS from UTMOS. A formal study also needs a declared sampling/listener protocol.

Outputs: `corpus.jsonl`, `data_provenance.json`, `corpus_asr_audit.jsonl`, `runs/<name>/train.jsonl`, `validation.jsonl`, `last.pt`, `best.pt`, `student_deploy.pt`, `parameters.json`, `scores/<name>/per_sample.csv`, `summary.json`, `benchmark_wavs/<name>/timings.jsonl`, adapted speaker similarity, and `comparison.csv`. All are under `ROOT` on the selected persistent disk. The notebook never uploads or publishes these files.

In [ ]:
if RUN_BENCHMARK:
    import random
    rng = random.Random(2026)
    manifest = mp.read_jsonl(ROOT / "benchmark_manifest.jsonl")
    unique = list({mp.text_key(r["text"]): r for r in manifest}.values())
    chosen = rng.sample(unique, min(30, len(unique)))
    for name, _, _ in EXPERIMENTS:
        folder = ROOT / "listening" / name
        folder.mkdir(parents=True, exist_ok=True)
        ratings, key = [], []
        for i, row in enumerate(chosen):
            a, b = ("teacher_maichi", name) if rng.random() < 0.5 else (name, "teacher_maichi")
            for label, run_name in [("A", a), ("B", b)]:
                shutil.copy2(ROOT / "benchmark_wavs" / run_name / row["output_wav"], folder / f"{i:03d}_{label}.wav")
            ratings.append({"pair": i, "text": row["text"], "A_naturalness_1_to_5": "", "B_naturalness_1_to_5": "",
                            "A_tone_errors": "", "B_tone_errors": "", "preferred": "", "notes": ""})
            key.append({"pair": i, "id": row["id"], "A": a, "B": b})
        pd.DataFrame(ratings).to_csv(folder / "ratings.csv", index=False)
        pd.DataFrame(key).to_csv(folder / "key_keep_separate.csv", index=False)
print("Results directory:", ROOT)


## Scope and interpretation

- A runnable pipeline and three real-WAV training steps do not prove a high-quality distilled student. Report full-run measurements only after running them.
- This version does not implement the Teacher in the Loop decoder, shared excitation or teacher iSTFT-head losses. It does not claim those paper results transfer to ZeroTTS.
- Student output is 24 kHz and uses text characters with Vietnamese diacritics, not a verified Vietnamese phoneme frontend. Inspect OOVs, code-switching, numbers and rare tones.
- Pitch supervision is a proposed adaptation. Compare pitch/no-pitch under the same data, backbone size, steps and seed before claiming a benefit; repeat full experiments across seeds for research conclusions.
- 137 public items are a compact test set. Select hyperparameters on the independent validation set, keep the test set frozen, and add unseen human-reviewed material before a paper claim.

Sources: [ZeroTTS](https://github.com/zeroweight-ai/ZeroTTS), [ZeroSpeech](https://huggingface.co/datasets/zeroweight-ai/ZeroSpeech), [ZeroBench-TTS](https://huggingface.co/datasets/zeroweight-ai/ZeroBench-TTS), [VITS](https://github.com/jaywalnut310/vits), [Teacher in the Loop](https://github.com/MithrilMan/teacher-in-the-loop).